# DNA-DetectLLM — Live Demo (Gradio UI)

**IE 663 | Team Eternal — Tanmay Mandaliya (22B1037)**

To replicate: Run this notebook on **Kaggle T4×2**. It launches a Gradio web app with a **public URL** you can open in any browser.

**Features:**
- Paste any text → get Human/AI verdict
- Score breakdown (PPL, X-PPL, mutation rate)
- Token mutation map (colored visualization)
- Repair curve (step-by-step perplexity drop)

In [1]:
!pip install transformers accelerate bitsandbytes gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.8 MB/s eta 0:00:00:00:0100:01


In [2]:
import shutil

src = "/kaggle/input/datasets/mandaliyatanmay/dna-detectllm/dna_detectllm"
shutil.copytree(src, "dna_detectllm", dirs_exist_ok=True)

!ls dna_detectllm/

detector.py  __init__.py  metrics.py  utils.py


In [3]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

import torch
import torch.nn.functional as F
import numpy as np
import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from dna_detectllm import DetectLLM
from dna_detectllm.metrics import perplexity, sum_perplexity, entropy
from dna_detectllm.detector import DEVICE_1, DEVICE_2

print("Loading models...")
detector = DetectLLM(
    observer_name_or_path="tiiuae/falcon-7b",
    performer_name_or_path="tiiuae/falcon-7b-instruct",
    use_4bit=True,
    max_token_observed=512,
    mode="accuracy",
)
print(f"Models loaded. GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB, "
      f"GPU 1: {torch.cuda.memory_allocated(1)/1e9:.1f} GB")

Loading models...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Models loaded. GPU 0: 4.9 GB, GPU 1: 4.9 GB


In [7]:
import time, random as _random

@torch.inference_mode()
def get_logits(text):
    encodings = detector._tokenize([text])
    obs_logits = detector.observer_model(**encodings.to(DEVICE_1)).logits
    perf_logits = detector.performer_model(**encodings.to(DEVICE_2)).logits
    return encodings.to(DEVICE_1), obs_logits.to(DEVICE_1), perf_logits.to(DEVICE_1)


@torch.inference_mode()
def get_token_analysis(text):
    """Get per-token analysis: actual vs ideal token, probabilities, mutation status."""
    encodings = detector._tokenize([text])
    perf_logits = detector.performer_model(**encodings.to(DEVICE_2)).logits.float().cpu()
    input_ids = encodings.input_ids.cpu()
    attn = encodings.attention_mask.cpu()

    shifted = perf_logits[0, :-1, :]
    mask = attn[0, 1:].float()
    actual = input_ids[0, 1:]
    ideal = shifted.argmax(dim=-1)
    probs = F.softmax(shifted, dim=-1)

    ce_actual = F.cross_entropy(shifted, actual, reduction="none")
    ce_ideal = F.cross_entropy(shifted, ideal, reduction="none")

    tokens = []
    for i in range(len(actual)):
        tok_id = actual[i].item()
        ideal_id = ideal[i].item()
        is_mut = tok_id != ideal_id
        tokens.append({
            "pos": i,
            "text": detector.tokenizer.decode([tok_id]),
            "ideal_text": detector.tokenizer.decode([ideal_id]),
            "actual_prob": probs[i, tok_id].item(),
            "ideal_prob": probs[i, ideal_id].item(),
            "is_mutation": is_mut,
            "gap": probs[i, ideal_id].item() - probs[i, tok_id].item(),
            "ce_actual": ce_actual[i].item(),
            "ce_ideal": ce_ideal[i].item(),
            "active": mask[i].item() > 0,
        })
    return tokens, mask, ce_actual, ce_ideal


def _esc(s):
    """HTML-escape a string."""
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")


def build_mutation_map_html(tokens):
    """Compact mutation map for the right sidebar."""
    n_mut = sum(1 for t in tokens if t["is_mutation"])
    n_total = len(tokens)
    mut_rate = n_mut / n_total * 100 if n_total else 0

    html = '<div style="font-family:\'Courier New\',monospace; line-height:2.2; padding:10px; background:#1a1a2e; border-radius:8px; font-size:12px;">'
    for t in tokens:
        tok = _esc(t["text"]) if t["text"].strip() else "␣"
        if t["is_mutation"]:
            intensity = min(1.0, t["gap"] * 3)
            r = int(180 + 75 * intensity)
            g = int(60 - 30 * intensity)
            b = int(60 - 30 * intensity)
            tip = _esc(f"Pos {t['pos']}: '{t['text']}' (p={t['actual_prob']:.3f}) → expected '{t['ideal_text']}' (p={t['ideal_prob']:.3f})")
            html += f'<span title="{tip}" style="background:rgb({r},{g},{b}); color:#fff; padding:2px 4px; margin:0 1px; border-radius:3px; border-bottom:2px solid #ff1744; cursor:help; font-weight:bold;">{tok}</span>'
        else:
            html += f'<span style="background:#1b5e20; color:#a5d6a7; padding:2px 4px; margin:0 1px; border-radius:3px;">{tok}</span>'
    html += '</div>'
    html += f'<div style="margin-top:8px; padding:8px; background:#f5f5f5; border-radius:6px; font-size:12px; color:#333;">'
    html += f'<b style="color:#000;">Mutations:</b> {n_mut}/{n_total} ({mut_rate:.1f}%) &nbsp;'
    html += '<span style="background:#1b5e20; color:#a5d6a7; padding:2px 6px; border-radius:3px; font-size:11px;">match</span> '
    html += '<span style="background:#b71c1c; color:#fff; padding:2px 6px; border-radius:3px; font-size:11px;">mutation</span>'
    html += '</div>'
    return html


def order_mutations(tokens, strategy):
    """Get ordered list of mutation tokens for a given strategy."""
    mutations = [t for t in tokens if t["is_mutation"] and t["active"]]
    if strategy == "high_to_low":
        return sorted(mutations, key=lambda t: -t["gap"])
    elif strategy == "low_to_high":
        return sorted(mutations, key=lambda t: t["gap"])
    elif strategy == "random":
        m = mutations.copy()
        _random.shuffle(m)
        return m
    else:  # sequential
        return mutations


def build_repair_sequence_html(tokens, repaired_set, current_pos=None):
    """
    Build the token sequence HTML for the repair animation.
    - repaired_set: set of positions already repaired (shown green)
    - current_pos: position currently being repaired (shown with yellow glow)
    - remaining mutations: shown red
    - non-mutations: shown grey/neutral
    """
    html = '<div style="font-family:\'Courier New\',monospace; line-height:2.4; padding:12px; background:#0d1117; border-radius:8px; font-size:13px; margin-bottom:12px;">'
    for t in tokens:
        pos = t["pos"]
        tok = _esc(t["text"]) if t["text"].strip() else "␣"

        if pos == current_pos:
            # Currently being repaired — bright yellow highlight with glow
            html += f'<span style="background:#f9a825; color:#000; padding:3px 5px; margin:0 1px; border-radius:4px; font-weight:bold; box-shadow:0 0 8px #fdd835, 0 0 16px #fdd835;">{tok}</span>'
        elif pos in repaired_set:
            # Already repaired — green
            html += f'<span style="background:#2e7d32; color:#c8e6c9; padding:3px 5px; margin:0 1px; border-radius:4px;">{tok}</span>'
        elif t["is_mutation"]:
            # Unrepaired mutation — red
            intensity = min(1.0, t["gap"] * 3)
            r = int(180 + 75 * intensity)
            g = int(60 - 30 * intensity)
            b = int(60 - 30 * intensity)
            html += f'<span style="background:rgb({r},{g},{b}); color:#fff; padding:3px 5px; margin:0 1px; border-radius:4px; border-bottom:2px solid #ff1744; font-weight:bold;">{tok}</span>'
        else:
            # Non-mutation — dark neutral
            html += f'<span style="background:#21262d; color:#8b949e; padding:3px 5px; margin:0 1px; border-radius:4px;">{tok}</span>'
    html += '</div>'
    return html


def build_repair_table_header():
    """Build the header for the repair steps table."""
    html = '<table style="width:100%; border-collapse:collapse; font-size:12px; font-family:monospace;">'
    html += '<tr style="background:#2c3e50; color:#fff;">'
    html += '<th style="padding:6px; text-align:center; color:#fff;">Step</th>'
    html += '<th style="padding:6px; text-align:center; color:#fff;">Pos</th>'
    html += '<th style="padding:6px; text-align:left; color:#fff;">Original</th>'
    html += '<th style="padding:6px; text-align:center; color:#fff;">→</th>'
    html += '<th style="padding:6px; text-align:left; color:#fff;">Ideal</th>'
    html += '<th style="padding:6px; text-align:center; color:#fff;">Gap</th>'
    html += '<th style="padding:6px; text-align:center; color:#fff;">PPL after</th>'
    html += '</tr>'
    return html


def build_repair_table_row(i, t, ppl_after):
    """Build one row of the repair steps table."""
    bg = "#fff3e0" if i % 2 == 0 else "#ffffff"
    html = f'<tr style="background:{bg}; color:#333;">'
    html += f'<td style="padding:5px 6px; text-align:center; font-weight:bold; color:#e65100;">{i+1}</td>'
    html += f'<td style="padding:5px 6px; text-align:center; color:#555;">{t["pos"]}</td>'
    html += f'<td style="padding:5px 6px; background:#ffcdd2; border-radius:3px; color:#333;"><b style="color:#b71c1c;">{_esc(t["text"])}</b> <span style="color:#999; font-size:11px;">p={t["actual_prob"]:.3f}</span></td>'
    html += f'<td style="padding:5px 6px; text-align:center; color:#333;">→</td>'
    html += f'<td style="padding:5px 6px; background:#c8e6c9; border-radius:3px; color:#333;"><b style="color:#1b5e20;">{_esc(t["ideal_text"])}</b> <span style="color:#999; font-size:11px;">p={t["ideal_prob"]:.3f}</span></td>'
    html += f'<td style="padding:5px 6px; text-align:center; color:#333;">{t["gap"]:.3f}</td>'
    html += f'<td style="padding:5px 6px; text-align:center; color:#1565c0; font-weight:bold;">{ppl_after:.4f}</td>'
    html += '</tr>'
    return html


def build_repair_curve_plot(ppl_curve, n_mutations, strategy):
    """Build the repair curve plot from a list of PPL values."""
    if len(ppl_curve) < 2:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.text(0.5, 0.5, "No mutations to repair", ha='center', va='center', fontsize=14)
        ax.set_axis_off()
        return fig

    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = range(len(ppl_curve))
    ax.plot(x, ppl_curve, 'o-', color='#1565c0', lw=2.5, markersize=6,
            markerfacecolor='white', markeredgecolor='#1565c0', markeredgewidth=2, zorder=5)
    ax.fill_between(x, ppl_curve, alpha=0.08, color='#1565c0')

    ppl_range = max(ppl_curve[0] - ppl_curve[-1], 0.01)
    ax.annotate(f"Start: {ppl_curve[0]:.3f}", xy=(0, ppl_curve[0]),
                xytext=(0.5, ppl_curve[0] + ppl_range * 0.18),
                fontsize=9, color='#c62828', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#c62828', lw=1.5))
    ax.annotate(f"End: {ppl_curve[-1]:.3f}", xy=(len(ppl_curve)-1, ppl_curve[-1]),
                xytext=(max(0, len(ppl_curve)-2), ppl_curve[-1] - ppl_range * 0.18),
                fontsize=9, color='#2e7d32', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#2e7d32', lw=1.5))

    drop = ppl_curve[0] - ppl_curve[-1]
    mid_y = (ppl_curve[0] + ppl_curve[-1]) / 2
    ax.annotate('', xy=(len(ppl_curve)-0.3, ppl_curve[-1]),
                xytext=(len(ppl_curve)-0.3, ppl_curve[0]),
                arrowprops=dict(arrowstyle='<->', color='#e65100', lw=2))
    ax.text(len(ppl_curve)-0.1, mid_y, f"Drop: {drop:.3f}",
            fontsize=9, color='#e65100', fontweight='bold', va='center')

    strategy_names = {"sequential": "Sequential (left→right)", "high_to_low": "High-to-Low gap",
                      "low_to_high": "Low-to-High gap", "random": "Random order"}
    ax.set_xlabel('Repair Step', fontsize=11)
    ax.set_ylabel('Avg Cross-Entropy (Perplexity)', fontsize=11)
    ax.set_title(f'Repair Curve — {n_mutations} mutations | {strategy_names.get(strategy, strategy)}',
                 fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, linestyle='--')
    fig.tight_layout()
    return fig


print("Helper functions ready.")

Helper functions ready.


In [8]:
def analyze_text(text, strategy):
    """
    Generator function: yields progressive updates for the repair animation.
    Each yield updates all 5 outputs simultaneously.
    Outputs: [repair_animation_html, repair_plot, score_breakdown_md, score_gauge_plot, right_panel_html]
    """
    if not text or len(text.strip()) < 20:
        yield "Please enter at least 20 characters.", None, "", None, ""
        return

    text = text.strip()

    # ---- Phase 1: Compute scores (right panel) ----
    encodings, obs_logits, perf_logits = get_logits(text)
    pad_id = detector.tokenizer.pad_token_id

    ppl_val = float(perplexity(encodings, perf_logits)[0])
    sum_ppl_val = float(sum_perplexity(encodings, perf_logits)[0])
    x_ppl_val = float(entropy(obs_logits, perf_logits, encodings, pad_id)[0])
    ppl_ideal = sum_ppl_val - ppl_val
    final_score = sum_ppl_val / (2 * x_ppl_val)

    verdict = "HUMAN-WRITTEN" if final_score > detector.threshold else "AI-GENERATED"
    confidence = abs(final_score - detector.threshold) / detector.threshold * 100
    verdict_color = "#2e7d32" if final_score > detector.threshold else "#c62828"

    # Score breakdown markdown
    breakdown_md = f"""### Verdict: **{verdict}**

| Component | Value | Meaning |
|---|---|---|
| PPL(s) | {ppl_val:.4f} | Surprise at actual tokens |
| PPL(ŝ∣s) | {ppl_ideal:.4f} | Surprise at ideal tokens |
| Sum-PPL | {sum_ppl_val:.4f} | Numerator of R(s) |
| X-PPL(s) | {x_ppl_val:.4f} | Denominator (normalizer) |
| **R(s)** | **{final_score:.4f}** | **Detection score** |
| Threshold τ | {detector.threshold:.4f} | R(s) > τ → Human |
| Confidence | {confidence:.1f}% | Distance from threshold |
"""

    # Score gauge plot
    fig_gauge, ax2 = plt.subplots(figsize=(5, 1.8))
    bar_color = verdict_color
    ax2.barh([0], [final_score], color=bar_color, height=0.5, edgecolor='white', alpha=0.85)
    ax2.axvline(detector.threshold, color='black', ls='--', lw=2.5,
                label=f'τ = {detector.threshold:.3f}')
    ax2.set_xlim(0, max(final_score * 1.3, detector.threshold * 1.3))
    ax2.set_yticks([])
    ax2.set_xlabel('R(s)', fontsize=10)
    ax2.set_title(f'{verdict} (R={final_score:.4f})', fontsize=11,
                  color=bar_color, fontweight='bold')
    ax2.legend(fontsize=9)
    fig_gauge.tight_layout()

    # ---- Phase 2: Token analysis ----
    tokens, mask_t, ce_actual_t, ce_ideal_t = get_token_analysis(text)

    # Right panel: mutation map + score breakdown combined
    mutation_map = build_mutation_map_html(tokens)
    right_html = mutation_map

    # ---- Phase 3: Repair animation (left panel) ----
    ordered = order_mutations(tokens, strategy)
    n_mutations = len(ordered)
    repaired_set = set()
    total_mask = mask_t.sum().item()
    ce_current = ce_actual_t.clone()

    # Initial state: show all tokens, no repairs yet
    initial_ppl = (ce_current * mask_t).sum().item() / total_mask
    ppl_curve = [initial_ppl]

    strategy_labels = {"sequential": "Sequential (left → right)", "high_to_low": "High-to-Low (biggest gap first)",
                       "low_to_high": "Low-to-High (smallest gap first)", "random": "Random order"}

    status_html = f'<div style="background:#263238; color:#eceff1; padding:10px 14px; border-radius:8px; margin-bottom:10px; font-size:13px;">'
    status_html += f'<b>Strategy:</b> {strategy_labels.get(strategy, strategy)} &nbsp;|&nbsp; '
    status_html += f'<b>Mutations to repair:</b> {n_mutations} &nbsp;|&nbsp; '
    status_html += f'<b>Starting PPL:</b> {initial_ppl:.4f}'
    status_html += '</div>'

    seq_html = build_repair_sequence_html(tokens, repaired_set)
    table_html = '<div style="max-height:350px; overflow-y:auto;">' + build_repair_table_header() + '</table></div>'

    repair_html = status_html + seq_html + table_html
    yield repair_html, None, breakdown_md, fig_gauge, right_html

    if n_mutations == 0:
        repair_html = status_html + seq_html
        repair_html += '<p style="color:#aaa; font-style:italic; padding:8px;">No mutations found — text matches model predictions perfectly.</p>'
        yield repair_html, None, breakdown_md, fig_gauge, right_html
        return

    # Build table rows progressively
    table_rows = ""
    for i, t in enumerate(ordered):
        pos = t["pos"]

        # Step A: highlight current token (yellow glow)
        seq_html = build_repair_sequence_html(tokens, repaired_set, current_pos=pos)
        current_repair_html = status_html + seq_html
        current_repair_html += '<div style="max-height:350px; overflow-y:auto;">' + build_repair_table_header() + table_rows + '</table></div>'
        yield current_repair_html, None, breakdown_md, fig_gauge, right_html

        time.sleep(0.8)

        # Step B: repair the token — update CE, compute new PPL, add table row
        repaired_set.add(pos)
        ce_current[pos] = ce_ideal_t[pos]
        new_ppl = (ce_current * mask_t).sum().item() / total_mask
        ppl_curve.append(new_ppl)

        table_rows += build_repair_table_row(i, t, new_ppl)

        seq_html = build_repair_sequence_html(tokens, repaired_set)

        # Update status with progress
        progress_status = f'<div style="background:#263238; color:#eceff1; padding:10px 14px; border-radius:8px; margin-bottom:10px; font-size:13px;">'
        progress_status += f'<b>Strategy:</b> {strategy_labels.get(strategy, strategy)} &nbsp;|&nbsp; '
        progress_status += f'<b>Progress:</b> {i+1}/{n_mutations} repaired &nbsp;|&nbsp; '
        progress_status += f'<b>PPL:</b> {initial_ppl:.4f} → {new_ppl:.4f} (↓{initial_ppl - new_ppl:.4f})'
        progress_status += '</div>'

        current_repair_html = progress_status + seq_html
        current_repair_html += '<div style="max-height:350px; overflow-y:auto;">' + build_repair_table_header() + table_rows + '</table></div>'
        yield current_repair_html, None, breakdown_md, fig_gauge, right_html

        time.sleep(0.5)

    # Final: show repair curve plot
    fig_curve = build_repair_curve_plot(ppl_curve, n_mutations, strategy)

    # Final status
    final_status = f'<div style="background:#1b5e20; color:#c8e6c9; padding:10px 14px; border-radius:8px; margin-bottom:10px; font-size:13px;">'
    final_status += f'<b>✓ Repair complete!</b> &nbsp;|&nbsp; '
    final_status += f'<b>Strategy:</b> {strategy_labels.get(strategy, strategy)} &nbsp;|&nbsp; '
    final_status += f'<b>{n_mutations}</b> mutations fixed &nbsp;|&nbsp; '
    final_status += f'<b>PPL:</b> {initial_ppl:.4f} → {ppl_curve[-1]:.4f} (↓{initial_ppl - ppl_curve[-1]:.4f})'
    final_status += '</div>'

    final_repair_html = final_status + seq_html
    final_repair_html += '<div style="max-height:350px; overflow-y:auto;">' + build_repair_table_header() + table_rows + '</table></div>'

    yield final_repair_html, fig_curve, breakdown_md, fig_gauge, right_html

print("Analysis generator ready.")

Analysis generator ready.


## Launch Gradio App

Running the cell below starts the web UI. On Kaggle with `share=True`, you'll get a **public URL** like `https://xxxxx.gradio.live` — open it in your browser to use the demo.

The Kaggle T4×2 GPUs handle all the inference. You just interact through the browser.

In [ ]:
examples = [
    ["Large language models have transformed natural language processing by enabling zero-shot and few-shot learning across a wide range of tasks. These models leverage massive datasets and transformer architectures to generate human-like text that is often indistinguishable from human writing."],
    ["I was walking through the old part of town yesterday when I stumbled upon this tiny bookshop I'd never noticed before. The owner, an elderly woman with thick glasses, recommended a novel about a lighthouse keeper. I bought it on impulse and ended up reading half of it on the bus ride home."],
    ["The experimental results demonstrate that our proposed method achieves state-of-the-art performance on all benchmark datasets. We observe a significant improvement of 3.2% in F1 score compared to the previous best approach, while maintaining comparable computational efficiency."],
]

with gr.Blocks(
    title="DNA-DetectLLM Demo",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.purple,
        secondary_hue=gr.themes.colors.teal,
        neutral_hue=gr.themes.colors.slate,
        font=gr.themes.GoogleFont("Inter"),
    ),
    css="""
    .gradio-container { max-width: 100% !important; }
    """
) as demo:

    gr.Markdown("""
    # 🧬 DNA-DetectLLM — AI Text Detector
    **IE 663 | Team Eternal — Tanmay Mandaliya (22B1037) | IIT Bombay**

    Paste text below → see the **mutation-repair process** unfold step by step.
    Human text has more mutations and is harder to repair → higher R(s) score.
    """)

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(label="Input Text", placeholder="Paste any text here (at least 20 characters)...", lines=8)
        with gr.Column(scale=1):
            strategy_input = gr.Radio(
                choices=[("Sequential (left→right)", "sequential"), ("High-to-Low gap", "high_to_low"),
                         ("Low-to-High gap", "low_to_high"), ("Random order", "random")],
                value="sequential", label="Repair Strategy",
            )
            btn = gr.Button("🔬 Analyze", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("""### Repair Process
*Each mutation is repaired one at a time based on the chosen strategy.
**Red** = unrepaired | **Yellow** = repairing now | **Green** = repaired | **Grey** = non-mutation*""")
            repair_animation = gr.HTML(label="Token Sequence")
            repair_plot = gr.Plot(label="Repair Curve")

        with gr.Column(scale=1):
            gr.Markdown("### Detection Score")
            score_gauge = gr.Plot(label="Score Gauge")
            breakdown_out = gr.Markdown(label="Score Breakdown")
            gr.Markdown("### Mutation Map\n*Hover tokens for details. Darker red = bigger gap.*")
            right_panel = gr.HTML(label="Token Map")

    btn.click(
        fn=analyze_text,
        inputs=[text_input, strategy_input],
        outputs=[repair_animation, repair_plot, breakdown_out, score_gauge, right_panel],
    )

    gr.Examples(examples=examples, inputs=[text_input])

    gr.Markdown("---\n*[DNA-DetectLLM](https://arxiv.org/abs/2509.15550) (NeurIPS 2025) | Falcon-7B + Falcon-7B-Instruct | 4-bit NF4 on Kaggle T4×2*")

demo.launch(share=True)